# 01.9 Phase 1 Project: Tabular Classification

This notebook is the integrated mini-project for Phase 1. The goal is not to chase the strongest model; the goal is to connect the whole supervised-learning workflow in PyTorch. You will load data, split it, standardize features, build DataLoaders, define an MLP, train it, evaluate it, and inspect the result.

Treat this as a small project rather than a collection of isolated code cells. Each step creates an object or decision that later cells depend on.

## Learning Goals

After this notebook, you should be able to:

1. Complete a minimal end-to-end classification project.
2. Use `PyTorch` for tabular data.
3. Understand the train, validation, and test stages.
4. Monitor training with loss and accuracy.
5. Predict on new samples.
6. Build a foundation for larger projects later.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

## Load the Data

We use the `iris` dataset because it is small enough to be a good first complete project.


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()

print(df.head())
print()
print("shape =", df.shape)
print("target names =", iris.target_names)

## Split into Train, Validation, and Test

Here we split twice:

1. first split out the test set
2. then split a validation set from the training set

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

print("train shape =", X_train.shape)
print("val shape =", X_val.shape)
print("test shape =", X_test.shape)

## Standardize Features

The scaler is fit only on the training set, then applied to validation and test sets.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled[:2])

## Convert to Tensors and Build DataLoaders

This step connects the `sklearn / pandas` world to the `PyTorch` world.


In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## Define the Model

The Iris dataset has 4 numeric input features, so the first linear layer must accept 4 values per sample. The task has 3 possible classes, so the final layer must output 3 logits per sample. The hidden dimension is a design choice: it controls the size of the intermediate representation the model learns before producing class scores.

In [ ]:
class IrisMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 3),
        )

    def forward(self, x):
        return self.net(x)


model = IrisMLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

print(model)

## Define Training and Evaluation Functions

We wrap training logic into functions so the project stays clearer.


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## Start Training

We train for 40 epochs here.


In [ ]:
history = []

for epoch in range(1, 41):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

In [ ]:
history_df = pd.DataFrame(history)
print(history_df.tail())

## Evaluate on the Test Set

Training and tuning use the validation set; the test set is used only for the final evaluation.


In [ ]:
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

test_acc = accuracy_score(all_targets.numpy(), all_preds.numpy())
cm = confusion_matrix(all_targets.numpy(), all_preds.numpy())

print("test accuracy =", test_acc)
print("confusion matrix =\n", cm)

## Run Inference on New Samples

One practical value of a trained model is making predictions on new data.


In [ ]:
new_samples = pd.DataFrame(
    {
        "sepal length (cm)": [5.0, 6.5],
        "sepal width (cm)": [3.4, 3.0],
        "petal length (cm)": [1.5, 5.5],
        "petal width (cm)": [0.2, 2.0],
    }
)

new_scaled = scaler.transform(new_samples)
new_tensor = torch.tensor(new_scaled, dtype=torch.float32)

with torch.no_grad():
    logits = model(new_tensor)
    preds = logits.argmax(dim=1)

pred_names = [iris.target_names[i] for i in preds.tolist()]
print(preds)
print(pred_names)

## Mini Exercises

These exercises are meant to encourage small active experiments.


In [ ]:
# Exercise 1
#
# Experiment with model capacity.
#
# Change the hidden width from 16 to 32, train again, and compare validation
# accuracy. When you compare, do not only ask whether the final number changed.
# Also ask whether training accuracy and validation accuracy moved together or
# whether the wider model started to overfit.

Exercise 1 Reference Note

This is an experiment exercise, so the exact result depends on the run. Compare both validation accuracy and the train/validation gap; a wider model can improve capacity, but it can also overfit.

In [ ]:
# Exercise 2
#
# Experiment with the optimizer.
#
# Change the optimizer from Adam to SGD and observe training speed and final
# validation accuracy. Keep the rest of the setup as similar as possible so the
# optimizer is the main thing being compared.

Exercise 2 Reference Note

Adam usually converges faster with less tuning. SGD can work well, but it is more sensitive to learning rate and may need momentum or more epochs.

In [ ]:
# Exercise 3
#
# Answer in one or two full sentences:
# Why should the scaler be fit only on the training set?
#
# Your answer should mention data leakage and why validation/test statistics
# should not influence preprocessing learned before evaluation.

Exercise 3 Reference Answer

The scaler should be fit only on the training set to avoid leaking validation or test statistics into preprocessing.

## Summary

At this point, you have completed a minimal but complete PyTorch tabular classification project. The workflow connected data loading, dataset splitting, feature preprocessing, TensorDataset and DataLoader creation, model definition, a training loop, validation, test evaluation, and confusion-matrix inspection.

The most important lesson is that these steps form one system. A preprocessing decision changes model inputs. A model output shape must match the loss. A validation metric helps guide experiments, while the test set should be saved for the final estimate. This is the same structure that larger projects use, just at a smaller scale.